In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np



In [ ]:
df = pd.read_csv("../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail()

In [ ]:
# Fit to DNS with Ordinary Least Squares

"""
L(t) = 1
S(t) = 1-e^(-xt)/xt
C(t) = S(t) - e^(-xt) 

x = 0.0609
"""

maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
x = 0.0609

# matrix never changes 
# A = np.array([[]])

# for t in maturities:
#     L = 1
#     S = (1 - np.exp(-x * t)) / (x * t)
#     C = S - np.exp(-x * t)

#     np.append(A, [[L, S, C]], axis=0)

A = np.array([
    [
    1,
    (1 - np.exp(-x * t)) / (x * t),
    ((1 - np.exp(-x * t)) / (x * t)) - np.exp(-x * t),
    ]
    for t in maturities
])


rows = []

for row in df.itertuples():
    # get actual yields for each date
    y = [row._1, row._2, row._3, row._4, row._5, row._6, row._7, row._8, row._9, row._10, row._11]

    betas, *_ = np.linalg.lstsq(A, y)
    b1, b2, b3 = betas
    rows.append({'Date': row.Index, 'Beta 1': b1, 'Beta 2': b2, 'Beta 3': b3})

results_df = pd.DataFrame(rows).set_index("Date")
results_df.tail()


